# Análise exploratória: PCA e MDS com dados da ANP

## 1. Introdução do problema

Para quem trabalha no varejo de combustível, uma dúvida comum é se os estados e os meses têm comportamentos parecidos em termos de preço, volume, mistura de gasolina com etanol e oscilações mensais. Será que conseguimos enxergar grupos bem diferentes? Isso faz toda a diferença para pensar em estoque, mix de produtos e estratégia regional. Antes de criar modelos complexos, o ideal é mapear esses padrões e as exceções.

Aqui vamos usar **PCA** para resumir as variáveis numéricas e entender quais delas têm mais peso. Também vamos usar **MDS** para ver quais registros estão mais próximos ou distantes. Vale lembrar que esses métodos mostram associações e semelhanças, e não causa e efeito. As distâncias e os componentes nos ajudam a criar hipóteses, mas não provam que um preço específico gerou um volume x.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

import matplotlib.pyplot as plt
import pandas as pd
from src.data_preparation import FEATURES_NUMERICAS, padronizar_features, preparar_dados
from src.mds_analysis import aplicar_mds
from src.pca_analysis import aplicar_pca

## 2. Descrição das bases

Juntamos duas bases públicas da ANP:

* **Preços médios de revenda** por UF e mês (gasolina C e etanol hidratado), que é a série mensal estadual do site da ANP.
* **Vendas em volume** dos mesmos combustíveis, também por UF e mês, do dataset `vendas-combustiveis-m3`.

O período analisado pode ser configurado no código. Neste notebook, estamos usando dados de 2021 a 2025, acompanhando o relatório do script `run_analysis.py`. O resultado é uma tabela onde cada linha representa um par de UF e mês com seus respectivos preços e volumes alinhados.

## 3. Preparação dos dados

O processo de limpeza funciona assim: primeiro, padronizamos os nomes dos estados e siglas. Depois, filtramos apenas gasolina C e etanol hidratado. Pivotamos os preços e as vendas, fazemos o merge das duas bases (no código usamos só `mes_ano` e a sigla `uf` como chaves; nome do estado e região entram reconciliados depois com `combine_first`) e calculamos alguns indicadores. Esses indicadores incluem a participação do etanol, a proporção entre os volumes, o preço relativo e as taxas de variação mensal dentro de cada estado. Calculamos isso por estado para não misturar dados de regiões diferentes.

Qualquer valor ausente ou infinito é removido no final desse processo para não prejudicar os algoritmos do PCA e do MDS.

In [ ]:
dados = preparar_dados(periodo_inicio=2021, periodo_fim=2025)
dados.shape

## 4. Correção do problema no merge

Tivemos um problema com essas bases antes: as duas fontes escrevem o nome da região de um jeito diferente, como "Centro Oeste" e "Centro-Oeste". Se o merge exigesse texto idêntico do estado (`uf_nome`) ou da região como chave, várias combinações UF-mês acabavam de fora só por causa de cadastro inconsistente nas fontes, não por falta de dado útil no mercado.

Para resolver isso, padronizamos a região com `normalizar_regiao` e no **merge** usamos só **`mes_ano`** e a **sigla `uf`**. `uf_nome` e região entram reconciliadas depois com `combine_first`. Assim recuperamos algo em torno de **240 registros** que perdíamos antes (DF, GO, MS e MT quando o Centro-Oeste não batia entre as tabelas).

## 5. Features analisadas

### Seleção das variáveis

As variáveis não foram escolhidas ao acaso. Preços e preços relativos nos ajudam a comparar a competitividade de cada região. Os volumes em m³ mostram o tamanho real da demanda. A participação do etanol e a proporção etanol/gasolina mostram o peso do biocombustível no consumo geral.

É importante notar que as colunas `participacao_etanol` e `razao_etanol_gasolina` são parecidas. As duas medem o espaço do etanol, mas de formas diferentes. Mesmo assim, o PCA pode acabar dando um peso forte para essa questão de "etanol vs gasolina" justamente por essa similaridade. Isso é uma característica da nossa escolha de variáveis e não um erro nos dados.

| Dimensão | Features | Motivo |
|---|---|---|
| Preço | `preco_medio_gasolina_c`, `preco_relativo_etanol_gasolina`, `variacao_preco_gasolina_c` | Mostra o custo para o consumidor e a posição do etanol. A variação mensal indica mudanças ao longo do tempo. |
| Volume de mercado | `volume_gasolina_c_m3`, `volume_etanol_hidratado_m3` | Mostra o tamanho absoluto da demanda. Isso pode destacar muito os estados maiores, mas falaremos disso adiante. |
| Substituição | `participacao_etanol`, `razao_etanol_gasolina` | O peso do etanol no consumo total em relação à gasolina. |
| Dinâmica mensal | `variacao_volume_gasolina_c` | Mostra como as vendas de gasolina oscilam mês a mês em cada estado. |

Abaixo está a lista exata que usamos no projeto (a mesma do arquivo `src/data_preparation.py`):

In [ ]:
FEATURES_NUMERICAS

Dando uma olhada em como os dados ficam antes da padronização:

In [ ]:
dados[
    [
        "mes_ano",
        "uf",
        "regiao",
        "preco_medio_gasolina_c",
        "volume_gasolina_c_m3",
        "volume_etanol_hidratado_m3",
        "participacao_etanol",
        "variacao_volume_gasolina_c",
    ]
].head()

## 6. Padronização

Como temos preços em reais, volumes em metros cúbicos e proporções em porcentagem, não dá para comparar tudo diretamente. Algoritmos de distância dão muito peso para números grandes se não fizermos nada.

Por isso, aplicamos o **`StandardScaler`**. Ele ajusta as colunas para que todas tenham média zero e desvio padrão igual a um. Isso deixa tudo na mesma escala para rodarmos o PCA e o MDS. Claro que estados muito grandes ainda vão aparecer como extremos se tiverem um perfil muito diferente, mas pelo menos evitamos que uma variável domine o modelo só por estar na casa dos milhões.

In [ ]:
dados_padronizados, _scaler = padronizar_features(dados)
dados_padronizados.describe().round(3)

## 7. PCA: aplicação

### Como estamos usando o método

O PCA pega nossas oito variáveis e cria dois eixos principais. O primeiro eixo tenta capturar o máximo de diferença (variância) entre os dados. O segundo pega o que sobrou, seguindo uma direção totalmente diferente do primeiro.

Na prática, vamos usar isso para plotar os estados e meses num gráfico 2D, ver o quanto esses dois eixos explicam dos nossos dados e olhar as cargas (os pesos das variáveis). Lembre que resumir oito colunas em duas sempre perde alguma informação. Use o gráfico bidimensional como um bom resumo visual, não como a verdade absoluta.

In [ ]:
pca_df, cargas, variancia, _pca_full = aplicar_pca(dados_padronizados)
display(variancia)
display(cargas.sort_values("PC1", key=abs, ascending=False))

Vamos plotar as cargas. Esses gráficos de barra mostram o peso direto que cada variável tem na construção do Componente 1 (PC1) e do Componente 2 (PC2).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col, titulo in (
    (axes[0], "PC1", "Cargas, componente 1"),
    (axes[1], "PC2", "Cargas, componente 2"),
):
    s = cargas[col].sort_values(key=abs)
    ax.barh(s.index.astype(str), s.values, color="steelblue")
    ax.axvline(0.0, color="black", linewidth=0.7)
    ax.set_title(titulo)
plt.tight_layout()
plt.show()

## 8. Interpretação das cargas do PCA

As cargas mostram o quanto cada variável influencia os componentes criados. Um valor alto (seja ele positivo ou negativo) significa muito peso. Novamente, isso não indica causa e efeito.

No **primeiro componente (PC1)**, variáveis como `participacao_etanol`, `razao_etanol_gasolina`, os volumes e o `preco_relativo_etanol_gasolina` têm bastante peso. Esse eixo agrupa os dados pelo **mix de consumo e preço do etanol**. Estados onde o etanol é muito forte tendem a se afastar dos estados onde a gasolina domina.

No **segundo componente (PC2)**, a história muda. As variáveis que mais pesam são a **variação de preço e de volume da gasolina**, junto com o volume total de gasolina. Esse eixo parece focar nas **oscilações e choques de curto prazo**, misturando a dinâmica mensal com o patamar de preços.

**Atenção:** o PCA apenas agrupa correlações matemáticas. Não podemos usar isso para justificar políticas de impostos ou culpar um preço isolado pelo comportamento do mercado.

### Projeção dos dados (PC1 x PC2)

No gráfico abaixo, cada ponto é um mês específico de um estado. A cor dos pontos representa a participação do etanol só para facilitar a leitura. Poderíamos pintar por região do país, mas o mix de combustível deixa os grupos bem mais claros neste cenário.

In [ ]:
pca_plot = dados.join(pca_df)
ax = pca_plot.plot.scatter(
    x="PC1",
    y="PC2",
    c="participacao_etanol",
    colormap="viridis",
    figsize=(8, 6),
)
ax.set_title("PCA: participação do etanol (cor)")
ax.set_xlabel(f"PC1 ({variancia.loc[0, 'variancia_explicada']:.1%})")
ax.set_ylabel(f"PC2 ({variancia.loc[1, 'variancia_explicada']:.1%})")
plt.show()

## 9. Outliers no PCA

Para achar os pontos mais extremos, podemos calcular a distância deles até o centro do gráfico (a distância euclidiana da origem). Os registros no topo dessa lista são os mais fora da curva no nosso resumo 2D. Geralmente, são casos de estados com volumes enormes ou meses com comportamento muito atípico.

In [ ]:
pca_plot = dados.join(pca_df)
pca_plot["distancia_pca"] = (pca_plot["PC1"] ** 2 + pca_plot["PC2"] ** 2) ** 0.5
cols = [
    "mes_ano",
    "uf",
    "regiao",
    "preco_medio_gasolina_c",
    "volume_gasolina_c_m3",
    "participacao_etanol",
    "variacao_volume_gasolina_c",
    "PC1",
    "PC2",
    "distancia_pca",
]
pca_plot.sort_values("distancia_pca", ascending=False)[cols].head(12)

### O problema dos volumes absolutos

Como estamos usando o volume total em metros cúbicos cru, estados muito populosos e com grandes frotas (como São Paulo) sempre vão aparecer nos extremos. O PCA não está penalizando paulistas, eles simplesmente operam em uma escala muito maior que o resto do país. O problema aqui é que o tamanho da frota acaba se misturando com as diferenças de perfil de consumo.

Uma forma de melhorar isso em futuras iterações seria aplicar logaritmo nos volumes ou usar dados de consumo por habitante. Assim conseguiríamos isolar melhor os perfis de consumo independentemente do tamanho do estado.

## 10. MDS: aplicação

### Entendendo a abordagem

O MDS serve para compararmos os resultados de outra forma. Enquanto o PCA busca variância, o MDS tenta posicionar os pontos num gráfico 2D de forma que as distâncias entre eles sejam o mais parecidas possível com as distâncias reais levando em conta as oito colunas padronizadas originais. O foco aqui é descobrir **quem é vizinho de quem**.

A vantagem é que os grupos ficam muito visíveis. A desvantagem é que os eixos do MDS não têm um significado claro como "aqui é preço e ali é volume". Para o gráfico não ficar pesado, rodamos o modelo com uma amostra de até 800 pontos. O valor de "stress" que aparece no log ajuda a avaliar a qualidade matemática do ajuste.

In [ ]:
mds_df, mds = aplicar_mds(dados_padronizados, max_registros=800)
print(f"Stress MDS: {mds.stress_:.2f}")
mds_df.head()

### Gráfico do MDS

Mantivemos a cor baseada na participação do etanol para facilitar a leitura. O que importa olhar aqui é a distância relativa entre os pontos e a formação das vizinhanças.

In [ ]:
mds_plot = dados.loc[mds_df.index].join(mds_df)
ax = mds_plot.plot.scatter(
    x="MDS1",
    y="MDS2",
    c="participacao_etanol",
    colormap="plasma",
    figsize=(8, 6),
)
ax.set_title("MDS: proximidade entre registros (cor = participação etanol)")
ax.set_xlabel("MDS1")
ax.set_ylabel("MDS2")
plt.show()

## 11. PCA vs MDS

O **PCA** é ótimo porque nos mostra exatamente o que está puxando os dados. Podemos olhar para o gráfico e justificar um agrupamento baseado nas cargas que calculamos. É um método que facilita a explicação de negócio.

Já o **MDS** ignora essa relação explícita com as variáveis e foca na similaridade total entre os pontos. Ele costuma confirmar os clusters do PCA, mas com algumas diferenças nas bordas dos grupos. Resumindo: use o PCA quando precisar explicar o motivo matemático dos grupos, e o MDS quando quiser apenas observar as vizinhanças e similaridades.

## 12. Limitações da análise

* **Sem causa e efeito:** Variáveis andando juntas não significa que uma causa a outra.
* **Resumo imperfeito:** Reduzir para duas dimensões sempre apaga detalhes. Decisões reais de negócio precisam olhar os dados mais a fundo.
* **Correlações óbvias:** A participação do etanol e a proporção etanol/gasolina dizem quase a mesma coisa. Isso pode dar um peso exagerado para o etanol na análise.
* **Efeito de escala:** O volume absoluto destaca estados grandes em vez de focar apenas no comportamento e nas escolhas de consumo.
* **Falta de contexto local:** Dados agregados mensais da ANP não mostram o impacto de uma frota específica, nível de renda ou eventos locais do estado.

## 13. Conclusão

Usar PCA e MDS em conjunto criou um mapa visual muito bom de como preço, volume e consumo de etanol aproximam ou separam os estados mês a mês. O PCA ajudou a entender o peso de cada variável na separação dos grupos, e o MDS confirmou os agrupamentos pela similaridade geral.

Claro que existem limitações, como a questão dos volumes absolutos e o foco exclusivo em séries agregadas por UF e mês, sem variáveis externas (renda, eventos locais, frota detalhada). Isso não mata o exercício: só indica caminhos para refinar depois. Para uma etapa exploratória com dados públicos da ANP, PCA e MDS cumprem bem o papel de levantar padrões e hipóteses antes de modelos ou decisões mais caras.